# 🔍 Multistage ReAct Reasoning Pipeline — Demo

This notebook walks through the **3-stage LLM reasoning pipeline** defined in
`rag/generation/reasoning_act.py`, step by step:

| Stage | LLM Call | Purpose |
|-------|----------|--------|
| 1 | `ReasonActAnalysisOutput` | Extract facts, candidates, mitigation/aggravation factors |
| 2 | `ReasonActLegalAnalysis` | Select offence, assess supporting articles, request additional law |
| 3 | `ReasonActFinalOutput` | Produce final verdict prediction with sentencing |

Between each LLM call, **retrieval steps** fetch law articles and similar past cases.

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 0 — Imports, environment, initialization
# ═══════════════════════════════════════════════════════════════════════
import json, os, sys
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv

# Ensure the repo root is on sys.path
REPO_ROOT = Path("/home/hieujayce/Downloads/complete_repo")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

# ── Core imports ──────────────────────────────────────────────────────
from rag.config import DEFAULT_MODEL_NAME, DEFAULT_DEVICE, DEFAULT_COLLECTION_NAME
from rag.core.law_retriever import LawClauseRetriever
from rag.evaluation.eval_utils import load_articles_index, _extract_gt_defendants
from rag.generation.reasoning_act import (
    extract_input_payload,
    build_query_text,
    doc_id_from_case,
    retrieve_candidate_articles,
    retrieve_supporting_articles,
    retrieve_similar_cases,
    retrieve_sentencing_calibration_cases,
    ensure_mandatory_supporting_assessments,
    _candidate_prompt,
    _legal_analysis_prompt,
    _final_prompt,
    _call_llm,
    _filter_new_law_signatures,
    _additional_law_query_signatures,
    retrieve_law_articles,
    _canonical_law_signature,
    _existing_law_coverage,
    MANDATORY_SUPPORTING_DIEU,
    DEFAULT_REASON_ACT_TRAIN_FIELDS,
    DEFAULT_SENTENCING_CALIBRATION_FIELDS,
)
from rag.generation.schemas import (
    ReasonActAnalysisOutput,
    ReasonActLegalAnalysis,
    ReasonActFinalOutput,
)
from rag.llm.providers import LLMProvider
from rag.runtime.retrieval import RetrievalRuntime, RetrievalRuntimeConfig

# ── Paths ───────────────────────────────────────────────────────────
TRAIN_DIR   = REPO_ROOT / "chunk" / "train"
TEST_DIR    = REPO_ROOT / "chunk" / "test"
LAW_JSON    = REPO_ROOT / "raw_law.json"
CASE_DB_DIR = REPO_ROOT / "output" / "reasoning_act_eval" / "case_db"

INPUT_FIELDS = ["THONG_TIN_CHUNG.Thong_Tin_Bi_Cao", "Synthetic_summary_2"]
QUERY_FIELDS = ["Synthetic_summary_2", "THONG_TIN_CHUNG.Thong_Tin_Bi_Cao"]

# ── LLM provider config ──────────────────────────────────────────────
PROVIDER   = LLMProvider.AISTUDIO
MODEL_NAME = "gemma-4-31b-it"   # change as needed

# ── Initialize heavy components once ─────────────────────────────────
print("Loading law retriever …")
law_retriever = LawClauseRetriever(LAW_JSON)

print("Building train article index …")
train_articles_index, train_skipped = load_articles_index(TRAIN_DIR)
print(f"  Index covers {len(train_articles_index)} docs  (skipped {len(train_skipped)})")

print("Connecting to case vector DB …")
case_runtime = RetrievalRuntime(
    RetrievalRuntimeConfig(
        model_name=DEFAULT_MODEL_NAME,
        device=DEFAULT_DEVICE,
        train_db_dir=str(CASE_DB_DIR),
        collection_name=DEFAULT_COLLECTION_NAME,
    )
)
print(f"  DB doc count = {case_runtime.train_doc_count()}")

# ── Load a sample test case ──────────────────────────────────────────
SAMPLE_FILE = sorted(TEST_DIR.glob("*.json"))[0]   # first file alphabetically
with open(SAMPLE_FILE, encoding="utf-8") as fh:
    case_data = json.load(fh)

doc_id = doc_id_from_case(case_data, SAMPLE_FILE.stem)
print(f"\n✅ Loaded sample: {SAMPLE_FILE.name}  (doc_id = {doc_id})")

Loading law retriever …
Building train article index …
  Index covers 4487 docs  (skipped 7)
Connecting to case vector DB …
  DB doc count = 20296

✅ Loaded sample: 01-02-2024-Khanh_Hoa-2ta1430918t1cvn.json  (doc_id = 01-02-2024-Khanh_Hoa-2ta1430918t1cvn)


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1 — Show the input: defendant info + synthetic summary
# ═══════════════════════════════════════════════════════════════════════
case_payload = extract_input_payload(case_data, INPUT_FIELDS)
query_text   = build_query_text(case_data, QUERY_FIELDS)
case_text    = "\n\n".join(case_payload.values())

print("── Input fields sent to the LLM ──────────────────────────")
for field, value in case_payload.items():
    print(f"\n🔹 [{field}]")
    # Pretty-print JSON-like fields, plain-print text
    try:
        parsed = json.loads(value)
        print(json.dumps(parsed, ensure_ascii=False, indent=2))
    except (json.JSONDecodeError, TypeError):
        print(value[:2000])

print("\n── Ground-truth verdict (for later comparison) ──────────")
gt_defendants = _extract_gt_defendants(case_data, only_blhs=True)
for d in gt_defendants:
    print(f"  Bị cáo: {d['Bi_Cao']}")
    print(f"  Tội danh: {d['Toi_Danh']}")
    print(f"  Phạt tù: {d['Phat_Tu']}")
    print(f"  Điều luật: {d['Applied_Law_Clauses']}")
    print()

── Input fields sent to the LLM ──────────────────────────

🔹 [THONG_TIN_CHUNG.Thong_Tin_Bi_Cao]
[
  {
    "Ho_Ten": "NTM",
    "Ngay_Sinh": "1997-07-22",
    "Noi_Cu_Tru": "Thôn XL2, xã VL, thành phố NT, tỉnh KH",
    "Nghe_Nghiep": "Không",
    "Trinh_Do_Van_Hoa": "9/12",
    "Dan_Toc": "Kinh",
    "Gioi_Tinh": "Nam",
    "Ton_Giao": "Không",
    "Quoc_Tich": "Việt Nam",
    "Hoan_Canh_Gia_Dinh": "con ông NTM (sinh năm 1964) và bà TTXD(sinh năm 1967); Vợ, con: Không",
    "Tien_An": "Không",
    "Tien_Su": "Không",
    "Ngay_Tam_Giam": "2023-08-06",
    "Trang_Thai_Co_Mat": "Có mặt tại phiên tòa"
  }
]

🔹 [Synthetic_summary_2]
[
  "Thưa luật sư, vào khoảng 14 giờ 00 phút ngày 05/8/2023, tôi có đi bộ đến bãi biển đối diện khách sạn Havana, 38 TP, phường LT, thành phố NT, tỉnh KH để tìm cơ hội trộm cắp. Tại đây, tôi thấy ông A và bà TTLV, hai khách du lịch Na Uy, đi tắm biển và để lại một chiếc ba lô cùng một túi màu hồng trên cát, nên tôi đã lén lút lấy trộm. Sau đó, tôi đến cửa hàng 

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 2 — Stage 1: Extract facts & candidate offences (LLM call 1)
# ═══════════════════════════════════════════════════════════════════════
system_1, user_1 = _candidate_prompt(doc_id, case_payload)

print("📤 Calling LLM — Stage 1 (fact extraction + candidate offences) …")
facts_and_candidates, usage_1 = _call_llm(
    provider=PROVIDER,
    model_name=MODEL_NAME,
    system_prompt=system_1,
    user_prompt=user_1,
    output_model=ReasonActAnalysisOutput,
    use_provider_fallback=True,
)

print("\n✅ Stage 1 complete.  Structured output:")
print(json.dumps(facts_and_candidates.model_dump(), ensure_ascii=False, indent=2))

📤 Calling LLM — Stage 1 (fact extraction + candidate offences) …

✅ Stage 1 complete.  Structured output:
{
  "facts": {
    "defendants": [
      "NTM"
    ],
    "conduct": "Vào khoảng 14 giờ 00 phút ngày 05/8/2023, NTM đã lén lút lấy trộm một chiếc ba lô và một túi màu hồng của ông A và bà TTLV (khách du lịch Na Uy) để lại trên cát tại bãi biển đối diện khách sạn Havana, thành phố NT, tỉnh KH. Tài sản lấy trộm gồm một chiếc điện thoại iPhone 14 Pro và 2.600.000 đồng tiền mặt (các tài sản khác đã bị vứt bỏ).",
    "dates": [
      "2023-08-05",
      "2023-08-06"
    ],
    "victims": [
      "Ông A",
      "Bà TTLV"
    ],
    "property_value": "Điện thoại iPhone 14 Pro trị giá 20.759.200 đồng và 2.600.000 đồng tiền mặt.",
    "harm": "Ông A và bà TTLV bị mất tài sản gồm điện thoại và tiền mặt.",
    "admissions": "NTM thành khẩn khai báo và hối hận về hành vi của mình.",
    "prior_convictions": "Không có tiền án, tiền sự.",
    "mitigation_signals": [
      "Thành khẩn khai báo",


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 3 — Retrieve law articles from raw_law.json
# ═══════════════════════════════════════════════════════════════════════

# 3a — Offence articles (from LLM-proposed candidates)
offence_articles = retrieve_candidate_articles(
    facts_and_candidates.candidates, law_retriever
)
found_offence_text = "\n\n".join(
    a.text or "" for a in offence_articles if a.found
)

# 3b — Supporting articles (mandatory + offence-specific)
supporting_articles = retrieve_supporting_articles(
    case_text=case_text,
    selected_offence_text=found_offence_text,
    law_retriever=law_retriever,
)

# ── Display offence articles ─────────────────────────────────────────
print("── Retrieved OFFENCE articles ────────────────────────────")
for a in offence_articles:
    status = "✅ found" if a.found else "❌ not found"
    print(f"  {a.signature:12s}  [{a.level or '—':5s}]  {status}")
    if a.found and a.text:
        # Show first 300 chars of the law text
        print(f"    {a.text[:300]}…" if len(a.text) > 300 else f"    {a.text}")
    print()

# ── Display supporting articles (skip mandatory for brevity) ────────
print("── Retrieved SUPPORTING articles (non-mandatory sample) ──")
non_mandatory = [a for a in supporting_articles if a.signature not in MANDATORY_SUPPORTING_DIEU]
display_list = non_mandatory if non_mandatory else supporting_articles[:3]
for a in display_list:
    status = "✅ found" if a.found else "❌ not found"
    print(f"  Điều {a.signature:5s}  [{a.level or '—':5s}]  {status}")
    if a.found and a.text:
        print(f"    {a.text[:300]}…" if len(a.text) > 300 else f"    {a.text}")
    print()

print(f"Total offence articles: {len(offence_articles)}")
print(f"Total supporting articles: {len(supporting_articles)}")
print(f"Mandatory supporting Điều: {list(MANDATORY_SUPPORTING_DIEU)}")

── Retrieved OFFENCE articles ────────────────────────────
  173           [dieu ]  ✅ found
    [dieu 173] Tội trộm cắp tài sản
[khoan 1] Người nào trộm cắp tài sản của người khác trị giá từ 2.000.000 đồng đến dưới 50.000.000 đồng hoặc dưới 2.000.000 đồng nhưng thuộc một trong các trường hợp sau đây, thì bị phạt cải tạo không giam giữ đến 03 năm hoặc phạt tù từ 06 tháng đến 03 năm:
[diem a] …

  249           [dieu ]  ✅ found
    [dieu 249] Tội tàng trữ trái phép chất ma túy
[khoan 1] Người nào tàng trữ trái phép chất ma túy mà không nhằm mục đích mua bán, vận chuyển, sản xuất trái phép chất ma túy thuộc một trong các trường hợp sau đây, thì bị phạt tù từ 03 năm đến 05 năm:
[diem a] Đã bị xử phạt vi phạm hành chính về hàn…

── Retrieved SUPPORTING articles (non-mandatory sample) ──
  Điều 46     [dieu ]  ✅ found
    [dieu 46] Các biện pháp tư pháp
[khoan 1] Biện pháp tư pháp đối với người phạm tội bao gồm:
[diem a] Tịch thu vật, tiền trực tiếp liên quan đến tội phạm;
[diem b] Trả

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 4 — Stage 2: Legal analysis (LLM call 2)
# ═══════════════════════════════════════════════════════════════════════
additional_articles = []   # starts empty; may grow if LLM requests more law

system_2, user_2 = _legal_analysis_prompt(
    doc_id=doc_id,
    facts_and_candidates=facts_and_candidates,
    offence_articles=offence_articles,
    additional_articles=additional_articles,
    supporting_articles=supporting_articles,
)

print("📤 Calling LLM — Stage 2 (legal analysis) …")
legal_analysis, usage_2 = _call_llm(
    provider=PROVIDER,
    model_name=MODEL_NAME,
    system_prompt=system_2,
    user_prompt=user_2,
    output_model=ReasonActLegalAnalysis,
    use_provider_fallback=True,
)

# Ensure all mandatory supporting assessments are present
legal_analysis.supporting_article_assessments = ensure_mandatory_supporting_assessments(
    legal_analysis.supporting_article_assessments,
    supporting_articles,
    case_text=case_text,
)

# ── Handle additional law round (if the LLM requested more articles) ─
requested_sigs = _additional_law_query_signatures(legal_analysis.additional_law_queries)
new_sigs = _filter_new_law_signatures(
    requested_sigs, offence_articles + supporting_articles + additional_articles
)
if new_sigs:
    print(f"  LLM requested additional law: {new_sigs}")
    additional_articles.extend(retrieve_law_articles(new_sigs, law_retriever))
    # Re-run stage 2 with the extra articles
    system_2b, user_2b = _legal_analysis_prompt(
        doc_id=doc_id,
        facts_and_candidates=facts_and_candidates,
        offence_articles=offence_articles,
        additional_articles=additional_articles,
        supporting_articles=supporting_articles,
        additional_law_round=1,
    )
    legal_analysis, usage_2b = _call_llm(
        provider=PROVIDER,
        model_name=MODEL_NAME,
        system_prompt=system_2b,
        user_prompt=user_2b,
        output_model=ReasonActLegalAnalysis,
        use_provider_fallback=True,
    )
    legal_analysis.supporting_article_assessments = ensure_mandatory_supporting_assessments(
        legal_analysis.supporting_article_assessments,
        supporting_articles,
        case_text=case_text,
    )
    print("  ✅ Re-ran Stage 2 with additional law.")
else:
    print("  (No additional law requested by the LLM.)")

print("\n✅ Stage 2 complete.  Structured output:")
print(json.dumps(legal_analysis.model_dump(), ensure_ascii=False, indent=2))

📤 Calling LLM — Stage 2 (legal analysis) …
  (No additional law requested by the LLM.)

✅ Stage 2 complete.  Structured output:
{
  "selected_offence": {
    "Dieu": "173",
    "Khoan": "1",
    "Diem": null,
    "offence_name": "Tội trộm cắp tài sản",
    "search_query": "Điều 173 Bộ luật Hình sự 2015 tội trộm cắp tài sản",
    "supporting_facts": "NTM lén lút lấy trộm một chiếc điện thoại iPhone 14 Pro trị giá 20.759.200 đồng và 2.600.000 đồng tiền mặt, tổng giá trị tài sản là 23.359.200 đồng, nằm trong khoảng từ 2.000.000 đồng đến dưới 50.000.000 đồng.",
    "rejection_or_downgrade_reason": null
  },
  "rejected_candidates": [
    {
      "Dieu": "249",
      "Khoan": null,
      "Diem": null,
      "offence_name": "Tội tàng trữ trái phép chất ma túy",
      "search_query": "Điều 249 Bộ luật Hình sự 2015 tàng trữ trái phép chất ma túy",
      "supporting_facts": "NTM thừa nhận dùng 500.000 đồng để mua một bịch ma túy 'cỏ'.",
      "rejection_or_downgrade_reason": "Thông tin không cu

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 5 — Retrieve similar past cases & sentencing calibration
# ═══════════════════════════════════════════════════════════════════════
selected_dieu = legal_analysis.selected_offence.Dieu

# Ensure the selected offence article is fully retrieved
selected_key = _canonical_law_signature(selected_dieu)
_, all_sigs = _existing_law_coverage(offence_articles + additional_articles)
if selected_key and selected_key not in all_sigs:
    offence_articles.extend(retrieve_law_articles([selected_dieu], law_retriever))

# 5a — Similar past cases (by factual profile)
print(f"🔎 Retrieving similar cases for Điều {selected_dieu} …")
similar_cases = retrieve_similar_cases(
    runtime=case_runtime,
    train_dir=TRAIN_DIR,
    train_articles_index=train_articles_index,
    query_text=query_text,
    selected_dieu=selected_dieu,
    exclude_doc_id=doc_id,
    broad_top_k=64,
    top_k=5,
)
print(f"  Found {len(similar_cases)} similar cases.")
if similar_cases:
    sc = similar_cases[0]
    print(f"\n── Sample similar case ───────────────────────────────────")
    print(json.dumps(sc.model_dump(), ensure_ascii=False, indent=2))

# 5b — Sentencing calibration cases (by mitigation/aggravation factors)
print(f"\n🔎 Retrieving sentencing calibration cases …")
print(f"  Mitigation factors: {facts_and_candidates.mitigation_factors}")
print(f"  Aggravation factors: {facts_and_candidates.aggravation_factors}")

sentencing_calibration_cases = retrieve_sentencing_calibration_cases(
    runtime=case_runtime,
    train_dir=TRAIN_DIR,
    train_articles_index=train_articles_index,
    mitigation_factors=facts_and_candidates.mitigation_factors,
    aggravation_factors=facts_and_candidates.aggravation_factors,
    selected_dieu=selected_dieu,
    exclude_doc_id=doc_id,
    top_k_per_factor=3,
    broad_top_k=64,
)
print(f"  Found {len(sentencing_calibration_cases)} calibration cases.")
if sentencing_calibration_cases:
    cc = sentencing_calibration_cases[0]
    print(f"\n── Sample calibration case ──────────────────────────────")
    print(json.dumps(cc.model_dump(), ensure_ascii=False, indent=2))

🔎 Retrieving similar cases for Điều 173 …


/home/hieujayce/Downloads/complete_repo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Loading BAAI/bge-m3 on cuda ...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 67605.44it/s]
/home/hieujayce/Downloads/complete_repo/rag/core/embeddings.py:209: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"  Embedding dim: {model.get_sentence_embedding_dimension()}")


  Embedding dim: 1024
  Found 5 similar cases.

── Sample similar case ───────────────────────────────────
{
  "doc_id": "28-12-2023-An_Giang-2ta1418141t1cvn",
  "matched_offence_article": "173",
  "matched_factual_profile": "Theo các tài liệu có trong hồ sơ vụ án và diễn biến tại phiên tòa, nội dung vu án được tóm tắt như sau: Khoảng 18 giờ ngày 26/3/2023, A M (người không rõ quốc tịch) điều khiển vỏ lãi Composite, màu xanh, có gắn máy xăng hiệu HINOTA 18,5HP chạy dọc theo sông K theo hướng từ xã C về xã L, thị xã T, tỉnh An Giang để tìm kiếm tài sản trôm cắp. Khi đến đoan sông thuộc khu vực ấp V, xã C, A M thấy chiếc sà lan của Trần Thị Thanh T1 và anh Huỳnh Ngọc N1 đang neo đậu, không người trông giữ nên điều khiển vỏ lãi cập vào sà lan, trèo lên cabin sà lan, sử dung dây chì tại sà lan mở ổ khóa cửa vào buồng lái lấy trôm số tiền 75.000.000 đồng và 01 điện thoại đi dộng nhãn hiệu Iphone 13 Promax màu xanh rồi xuống vỏ lãi điều khiển về chiếc bè của vợ chồng A M sinh sống, neo đâu t

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 6 — Stage 3: Final prediction (LLM call 3) + ground-truth comparison
# ═══════════════════════════════════════════════════════════════════════
system_3, user_3 = _final_prompt(
    doc_id=doc_id,
    case_payload=case_payload,
    facts_and_candidates=facts_and_candidates,
    legal_analysis=legal_analysis,
    offence_articles=offence_articles,
    additional_articles=additional_articles,
    supporting_articles=supporting_articles,
    similar_cases=similar_cases,
    sentencing_calibration_cases=sentencing_calibration_cases,
)

print("📤 Calling LLM — Stage 3 (final verdict prediction) …")
final_output, usage_3 = _call_llm(
    provider=PROVIDER,
    model_name=MODEL_NAME,
    system_prompt=system_3,
    user_prompt=user_3,
    output_model=ReasonActFinalOutput,
    use_provider_fallback=True,
)

print("\n✅ Stage 3 complete.  Final prediction:")
print(json.dumps(final_output.prediction.model_dump(), ensure_ascii=False, indent=2))

# ── Compare with ground truth ─────────────────────────────────────────
print("\n" + "═" * 70)
print("📊 COMPARISON: Prediction vs Ground Truth")
print("═" * 70)

for pred_def in final_output.prediction.defendants:
    name = pred_def.Bi_Cao
    # Find matching GT defendant
    gt_match = None
    for gt in gt_defendants:
        if gt["Bi_Cao"].strip().lower() == name.strip().lower():
            gt_match = gt
            break
    if gt_match is None and len(gt_defendants) == 1:
        gt_match = gt_defendants[0]

    print(f"\n── Bị cáo: {name} ──────────────────────────────────────")
    print(f"  {'':30s} {'PREDICTION':30s} {'GROUND TRUTH':30s}")
    print(f"  {'Tội danh':30s} {(pred_def.Toi_Danh or '—'):30s} {(gt_match or {}).get('Toi_Danh', '—'):30s}")
    print(f"  {'Phạt tù':30s} {(pred_def.Phat_Tu or '—'):30s} {(gt_match or {}).get('Phat_Tu', '—'):30s}")

    pred_clauses = sorted({f"{c.Dieu}-{c.Khoan or ''}" for c in pred_def.Applied_Law_Clauses if c.Dieu})
    gt_clauses = sorted((gt_match or {}).get("Applied_Law_Clauses", []))
    print(f"  {'Điều luật (pred)':30s} {', '.join(pred_clauses)}")
    print(f"  {'Điều luật (GT)':30s} {', '.join(gt_clauses)}")

    if pred_def.Phan_Tich_Phap_Ly:
        print(f"\n  📝 Legal reasoning (excerpt):")
        print(f"    {pred_def.Phan_Tich_Phap_Ly[:500]}")

if final_output.prediction.Xu_Ly_Vat_Chung:
    print(f"\n  Xử lý vật chứng: {final_output.prediction.Xu_Ly_Vat_Chung}")

print("\n" + "═" * 70)
print("Demo complete. All 3 LLM stages executed successfully.")

📤 Calling LLM — Stage 3 (final verdict prediction) …

✅ Stage 3 complete.  Final prediction:
{
  "defendants": [
    {
      "Bi_Cao": "NTM",
      "Phan_Tich_Phap_Ly": "Bị cáo NTM đã thực hiện hành vi lén lút chiếm đoạt tài sản của người khác gồm 01 điện thoại iPhone 14 Pro và 2.600.000 đồng, tổng giá trị tài sản là 23.359.200 đồng. Hành vi này đủ yếu tố cấu thành tội Trộm cắp tài sản theo quy định tại điểm a khoản 1 Điều 173 BLHS. Về tình tiết giảm nhẹ, bị cáo thành khẩn khai báo, ăn năn hối cải (điểm s khoản 1 Điều 51) và đã tự nguyện khắc phục hậu quả bằng cách trả lại toàn bộ tài sản cho bị hại (điểm b khoản 1 Điều 51). Bị cáo không có tiền án, tiền sự, nhân thân tốt. Do đó, bị cáo đủ điều kiện để được xem xét áp dụng hình phạt nhẹ hơn hoặc cho hưởng án treo theo Điều 65 BLHS.",
      "Toi_Danh": "Tội trộm cắp tài sản",
      "Applied_Law_Clauses": [
        {
          "Dieu": "173",
          "Khoan": "1",
          "Diem": null,
          "Tinh_tiet_ap_dung": "Chiếm đoạt tài sả